# fig1_new_region - Part 7/8\n\n自动拆分版本（按步骤执行）。\n包含统一 bootstrap 和共享模块导入。\n

In [ ]:
# AUTO_BOOTSTRAP_V2
from pathlib import Path
import sys
import os
import builtins
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "projects").exists():
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "projects").exists():
            ROOT = p
            break

NB_PATH = Path.cwd()
if "clone_motif" in str(NB_PATH):
    PROJECT_DIR = ROOT / "projects" / "clone_motif"
else:
    PROJECT_DIR = ROOT / "projects" / "our_multiregion_motif"

DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
OUT_FIG = PROJECT_DIR / "outputs" / "figures"
OUT_TABLE = PROJECT_DIR / "outputs" / "tables"
OUT_ARCH = PROJECT_DIR / "outputs" / "archives"

for d in [DATA_PROCESSED, OUT_FIG, OUT_TABLE, OUT_ARCH]:
    d.mkdir(parents=True, exist_ok=True)

SHARED_SRC = ROOT / "projects" / "shared" / "src"
if str(SHARED_SRC) not in sys.path:
    sys.path.append(str(SHARED_SRC))

from motif_common import combination, indices_for_region, union_indices_for_regions, p_to_star, format_p_decimal_3sig, sort_by_order, truncate_colormap

READ_EXT = {".csv", ".json", ".npy", ".pkl", ".xlsx"}
FIG_EXT = {".svg", ".png", ".pdf"}
TABLE_EXT = {".csv", ".xlsx"}


def _as_path(x):
    return Path(x) if isinstance(x, (str, os.PathLike)) else x


def resolve_read_path(path):
    p = _as_path(path)
    if not isinstance(p, Path):
        return path
    if p.is_absolute() or p.exists():
        return str(p)
    if p.suffix.lower() in READ_EXT:
        for c in [DATA_RAW / p.name, ROOT / p.name]:
            if c.exists():
                return str(c)
    return str(p)


def resolve_write_path(path):
    p = _as_path(path)
    if not isinstance(p, Path):
        return path
    if p.is_absolute():
        p.parent.mkdir(parents=True, exist_ok=True)
        return str(p)
    ext = p.suffix.lower()
    if ext in FIG_EXT:
        out = OUT_FIG / p.name
    elif ext in TABLE_EXT:
        out = OUT_TABLE / p.name
    elif ext == ".zip":
        out = OUT_ARCH / p.name
    elif ext == ".npy":
        out = DATA_PROCESSED / p.name
    else:
        out = PROJECT_DIR / p
    out.parent.mkdir(parents=True, exist_ok=True)
    return str(out)

if not hasattr(builtins, "_orig_open_codex"):
    builtins._orig_open_codex = builtins.open


def _open_patch(file, mode="r", *args, **kwargs):
    if isinstance(file, (str, os.PathLike)):
        if any(m in mode for m in ["r", "a"]):
            file = resolve_read_path(file)
        if any(m in mode for m in ["w", "a", "x"]):
            file = resolve_write_path(file)
    return builtins._orig_open_codex(file, mode, *args, **kwargs)


builtins.open = _open_patch

if not hasattr(np, "_orig_load_codex"):
    np._orig_load_codex = np.load
np.load = lambda file, *a, **k: np._orig_load_codex(resolve_read_path(file), *a, **k)

if not hasattr(np, "_orig_save_codex"):
    np._orig_save_codex = np.save
np.save = lambda file, arr, *a, **k: np._orig_save_codex(resolve_write_path(file), arr, *a, **k)

if not hasattr(pd, "_orig_read_csv_codex"):
    pd._orig_read_csv_codex = pd.read_csv
pd.read_csv = lambda f, *a, **k: pd._orig_read_csv_codex(resolve_read_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)

if not hasattr(pd, "_orig_read_excel_codex"):
    pd._orig_read_excel_codex = pd.read_excel
pd.read_excel = lambda f, *a, **k: pd._orig_read_excel_codex(resolve_read_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)

if not hasattr(pd.DataFrame, "_orig_to_csv_codex"):
    pd.DataFrame._orig_to_csv_codex = pd.DataFrame.to_csv


def _to_csv_patch(self, path_or_buf=None, *args, **kwargs):
    if isinstance(path_or_buf, (str, os.PathLike)):
        path_or_buf = resolve_write_path(path_or_buf)
    return pd.DataFrame._orig_to_csv_codex(self, path_or_buf, *args, **kwargs)


pd.DataFrame.to_csv = _to_csv_patch

if not hasattr(pd.DataFrame, "_orig_to_excel_codex"):
    pd.DataFrame._orig_to_excel_codex = pd.DataFrame.to_excel


def _to_excel_patch(self, excel_writer, *args, **kwargs):
    if isinstance(excel_writer, (str, os.PathLike)):
        excel_writer = resolve_write_path(excel_writer)
    return pd.DataFrame._orig_to_excel_codex(self, excel_writer, *args, **kwargs)


pd.DataFrame.to_excel = _to_excel_patch

try:
    import matplotlib.pyplot as plt
    if not hasattr(plt, "_orig_savefig_codex"):
        plt._orig_savefig_codex = plt.savefig
    plt.savefig = lambda f, *a, **k: plt._orig_savefig_codex(resolve_write_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)
except Exception:
    pass

print(f"[bootstrap] project={PROJECT_DIR.name} data={DATA_RAW}")


In [ ]:
# === 3. 在 UMAP 2D 上做层次聚类 ===
Z_link = linkage(X_umap, method="average", metric="euclidean")

# 比如想切成 3 类：
cluster_labels = fcluster(Z_link, t=4, criterion="maxclust")
print("Cluster labels (based on UMAP + hierarchical):")


# === 4. 画基于 UMAP 的树状图 ===
plt.figure(figsize=(2, 10))
dendrogram(
    Z_link,
    labels=region_names,#[" "]*len(region_names),#
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,               # 关闭按照距离分颜色
    above_threshold_color="k",       # 所有枝统一用黑色
    link_color_func=lambda k: "k",   # 再保险，所有连线都用黑色
)
# plt.title(f"Hierarchical clustering on UMAP embedding (thr={thr_to_use})")
plt.xlabel("Distance (on UMAP space)")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import optimal_leaf_ordering

# 用 UMAP 空间的距离做叶子排序
Y = pdist(X_umap,  metric="euclidean")
Z_opt = optimal_leaf_ordering(Z_link, Y)

plt.figure(figsize=(2.1, 10))
dendrogram(
    Z_opt,
    labels=[" "]*len(region_names),#
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="k",
    link_color_func=lambda k: "k",
)


plt.xlabel("Distance (on UMAP space)")

fig.patch.set_alpha(0)       # figure 背景透明
ax.set_facecolor("none")     # 坐标轴区域背景透明

plt.tight_layout()

# plt.savefig(
#     "region_dendrogram_umap.svg",
#     format="svg",
#     bbox_inches="tight",
#     transparent=True,         # 导出时保持透明
# )

plt.show()

In [ ]:
dendro = dendrogram(Z_opt, no_plot=True)
order = dendro["leaves"][::-1]    # 例如 [5, 2, 0, 7, ...]
# 如果想用聚类后的顺序，就改成 NZ_sorted / regions_sorted
# X_sorted: (n_regions, 13)，按 order 排好的 NZ 矩阵
X_sorted = NZ_mat[order, :]
regions_sorted = [region_names[i] for i in order]

# 只取前 12 维
X12 = X_sorted[:, :13]               # (n_regions, 12)

# 单位化（12 维）
norms = np.linalg.norm(X12, axis=1, keepdims=True) + 1e-8
Xn = X12 / norms                     # (n_regions, 12)

# 余弦相似度矩阵：S_ij = cos(theta_ij)
cos_sim = Xn @ Xn.T                  # (n_regions, n_regions)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(cos_sim, cmap="coolwarm", vmin=0, vmax=1)

ax.set_xticks(np.arange(len(regions_sorted)))
ax.set_yticks(np.arange(len(regions_sorted)))
ax.set_xticklabels(regions_sorted, rotation=90, fontsize=7)
ax.set_yticklabels(regions_sorted, fontsize=7)

ax.set_title(f"Cosine similarity of NZ-score (first 12 motifs, thr={thr_to_use})")
plt.colorbar(
    im,
    ax=ax,
    label="cosine similarity",
    shrink=0.25,     # 高度缩到原来的 1/4
)
plt.tight_layout()
# plt.savefig(
#     "region_cosine_simi.svg",
#     format="svg",
#     bbox_inches="tight",
#     transparent=True,         # 导出时保持透明
# )
plt.show()

In [ ]:
# === 5. 按 UMAP 层次聚类的顺序画原始 NZ-score 热图 ===
dendro = dendrogram(Z_opt, no_plot=True)
order = dendro["leaves"][::-1]

NZ_sorted = NZ_mat[order, :]
regions_sorted = [region_names[i] for i in order]

fig, ax = plt.subplots(figsize=(8, 8))

vmax = np.max(np.abs(NZ_sorted))
im = ax.imshow(NZ_sorted, aspect="auto", cmap="bwr", vmin=-vmax, vmax=vmax)

cbar = plt.colorbar(im, ax=ax, label="NZ-score")
ax.set_yticks(np.arange(len(regions_sorted)))
ax.set_yticklabels(regions_sorted, fontsize=8)
ax.set_xticks(np.arange(13))
ax.set_xticklabels([f"M{i}" for i in motif_ids], rotation=45)
ax.set_title(f"Region NZ-score heatmap (UMAP-based order, thr={thr_to_use})")

# 关键：背景透明
fig.patch.set_alpha(0)        # 整个 figure 背景透明
ax.set_facecolor("none")      # 坐标轴区域背景透明

plt.tight_layout()

# 保存为透明背景 SVG
fig.savefig(
    "region_NZ_heatmap_umap_order.svg",
    format="svg",
    bbox_inches="tight",
    transparent=True,
)

plt.show()

In [ ]:
from sklearn.cluster import KMeans

# ========= 手肘法：不同 k 的簇内平方和 =========
X_kmeans = X_umap   # 也可以改成原始 NZ_mat，看你想在哪个空间聚类

Ks = range(1, 10)   # k 从 1 到 9，你可以按需要改
inertias = []

for k in Ks:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
    )
    km.fit(X_kmeans)
    inertias.append(km.inertia_)  # 簇内平方和（越小说明类内越紧）

print("Ks:", list(Ks))
print("Inertias:", inertias)

plt.figure(figsize=(5, 4))
plt.plot(list(Ks), inertias, marker="o")
plt.xticks(list(Ks))
plt.xlabel("Number of clusters k")
plt.ylabel("Within-cluster sum of squares (inertia)")
plt.title("Elbow method on UMAP embedding")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# ========= 3. 在 UMAP 2D 上做 KMeans 聚类 =========
k = 4  # 你想要的聚类数，比如 3 或 4
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20,
)
km_labels = kmeans.fit_predict(X_umap)  # shape (n_regions,)

print(f"\n=== UMAP + KMeans 聚类结果 (k={k}) ===")
for r, c in zip(region_names, km_labels):
    print(f"{r:20s} -> cluster {c}")

# ========= 画图并保存为 SVG =========
fig, ax = plt.subplots(figsize=(5, 5))

for c in range(k):
    mask = (km_labels == c)
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1], label=f"Cluster {c}")

for i, r in enumerate(region_names):
    ax.text(X_umap[i, 0], X_umap[i, 1], r, fontsize=7, alpha=0.7)

ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.set_title(f"UMAP + KMeans clustering (k={k}, thr={thr_to_use})")
ax.legend(fontsize=8)

fig.tight_layout()

# 如果想要透明背景，可以加 transparent=True
fig.savefig(
    "umap_kmeans_k4.svg",
    format="svg",
    bbox_inches="tight",
    transparent=True,
)

plt.show()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ===== 你关心的区域 =====
region_list = ['PRE', 'ILA', 'FRP', 'PL']  # 示例
ref_region = 'MOp'                         # 参考区域

# 把 MOp 也加进来（避免重复）
all_regions = region_list + [ref_region]
all_regions = [r for r in all_regions if r in motif_results]  # 过滤掉不存在的

thr_to_use = 5  # 你实际用的 threshold，例如 5，对应 motif_results[region][5]

motif_ids = np.arange(1, 14)
motif_labels = [f"M{i}" for i in motif_ids]

# ===== 收集 NZ 和 log2 富集度 =====
nz_dict = {}
log2_enrich_dict = {}
info_dict = {}  # 存 N 和 density

eps = 1e-6

for region in all_regions:
    d_thr = motif_results[region].get(thr_to_use, None)
    if d_thr is None:
        print(f"[WARN] {region} 没有 thr={thr_to_use} 的结果，跳过")
        continue

    NZ = np.asarray(d_thr["NZ"], dtype=float)              # (13,)
    real = np.asarray(d_thr["motif_count"], dtype=float)   # (13,)
    mu = np.asarray(d_thr["er_mu"], dtype=float)           # (13,)

    # log2 富集度：log2(real) - log2(mu) = log2(real/mu)
    log2_real = np.log2(real + eps)
    log2_mu   = np.log2(mu   + eps)
    log2_enrich = log2_real - log2_mu

    nz_dict[region] = NZ
    log2_enrich_dict[region] = log2_enrich
    info_dict[region] = (d_thr["N"], d_thr["density"])

# 按顺序保留最终的 region 列表（保证都有效）
plot_regions = list(nz_dict.keys())
print("参与比较的区域:", plot_regions)

# ===== 画图：1) NZ-score 对比  2) log2 富集度对比 =====
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# 上面：NZ-score
ax1 = axes[0]
for region in plot_regions:
    NZ = nz_dict[region]
    N, dens = info_dict[region]
    label = f"{region} (N={N}, ρ={dens:.3f})"
    # 参考区 MOp 可以画粗一点
    if region == ref_region:
        ax1.plot(motif_ids, NZ, marker='o', linewidth=2.5, label=label)
    else:
        ax1.plot(motif_ids, NZ, marker='o', linewidth=1.5, label=label)

ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.set_ylabel("NZ-score")
ax1.set_title(f"NZ-score comparison vs {ref_region} (thr={thr_to_use})")
ax1.legend(fontsize=8, ncol=2)
ax1.grid(alpha=0.2)

# 下面：log2 富集度
ax2 = axes[1]
for region in plot_regions:
    log2_enrich = log2_enrich_dict[region]
    N, dens = info_dict[region]
    label = f"{region} (N={N}, ρ={dens:.3f})"
    if region == ref_region:
        ax2.plot(motif_ids, log2_enrich, marker='o', linewidth=2.5, label=label)
    else:
        ax2.plot(motif_ids, log2_enrich, marker='o', linewidth=1.5, label=label)

ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xlabel("Motif ID")
ax2.set_xticks(motif_ids)
ax2.set_xticklabels(motif_labels, rotation=0)
ax2.set_ylabel("log2(real/ER)")
ax2.set_title(f"log2 enrichment vs ER (thr={thr_to_use})")
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()